In [1]:

import ast
import os
from pathlib import Path

import pandas as pd
import spacy


def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "llm_pipeline").exists() and (candidate / "CV_analysis").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing llm_pipeline and CV_analysis.")


REPO_ROOT = find_repo_root()
CV_ROOT = REPO_ROOT / "CV_analysis"

# -------------------------------------------------
#                  CONFIG
# -------------------------------------------------
NAMES_CSV = CV_ROOT / "data/CV_names_1985.csv"
RUNS_CSV = REPO_ROOT / "llm_pipeline/outputs_export/cv_runs_fixed.csv"
POS_OUT_DIR = CV_ROOT / "data/pos_counts"
POS_OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATHS = {
    "openai": POS_OUT_DIR / "openai_cv_pos_counts.csv",
    "gemini": POS_OUT_DIR / "gemini_cv_pos_counts.csv",
    "qwen4B": POS_OUT_DIR / "qwen4B_cv_pos_counts.csv",
    "qwen8B": POS_OUT_DIR / "qwen8B_cv_pos_counts.csv",
}

# Backwards-compatible names for older exploratory cells.
OUT_OPENAI = OUT_PATHS["openai"]
OUT_GEMINI = OUT_PATHS["gemini"]

# load data
df_all = pd.read_csv(RUNS_CSV)
df_openai = df_all[df_all["provider"] == "openai"].copy()
df_gemini = df_all[df_all["provider"] == "google-gemini"].copy()
df_qwen4B = df_all[df_all["provider"].astype(str).str.contains("qwen3-4b", case=False, na=False)].copy()
df_qwen8B = df_all[df_all["provider"].astype(str).str.contains("qwen3-8b", case=False, na=False)].copy()

MODEL_RUNS = {
    "openai": df_openai,
    "gemini": df_gemini,
    "qwen4B": df_qwen4B,
    "qwen8B": df_qwen8B,
}

# load metadata
df_names = pd.read_csv(NAMES_CSV)

print("Loaded", RUNS_CSV, df_all.shape)
for model_name, model_df in MODEL_RUNS.items():
    print(f"{model_name}: {len(model_df)} rows")
print("Rows names:", len(df_names))

nlp = spacy.load("de_core_news_md")


Loaded /Users/charlotteleininger/Master/Consulting/llm_pipeline/outputs_export/cv_runs_fixed.csv (4800, 11)
openai: 1200 rows
gemini: 1200 rows
qwen4B: 1200 rows
qwen8B: 1200 rows
Rows names: 40


In [2]:
os.getcwd()

'/Users/charlotteleininger/Master/Consulting/CV_analysis/code/Structural'

In [2]:

# -------------------------------------------------
#                  HELPERS
# -------------------------------------------------
def flatten_json(j):
    items = []
    if isinstance(j, dict):
        # Do not count personal data in POS features, matching lexical TF extraction.
        if "01_persoenliche_daten" in j:
            j = {k: v for k, v in j.items() if k != "01_persoenliche_daten"}
        for v in j.values():
            items.extend(flatten_json(v))
    elif isinstance(j, list):
        for v in j:
            items.extend(flatten_json(v))
    elif isinstance(j, str):
        t = j.strip()
        if t:
            items.append(t)
    return items


def extract_json_snippet(text):
    if not isinstance(text, str):
        return None
    cleaned = text.strip()
    for fence in ("```json", "```JSON", "```"):
        if cleaned.startswith(fence):
            cleaned = cleaned[len(fence):].lstrip()
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3].rstrip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    snippet = cleaned[start:end + 1].strip()
    return snippet if snippet else None


def parse_response_payload(row):
    raw_json = row.get("response_json", "")
    if isinstance(raw_json, str) and raw_json.strip():
        try:
            return ast.literal_eval(raw_json)
        except Exception:
            pass
    fallback = row.get("response_text", "") or raw_json or ""
    snippet = extract_json_snippet(fallback)
    if snippet:
        return ast.literal_eval(snippet)
    raise ValueError("No parseable response JSON found")


def process_pos(df, df_names):
    df = df.copy()
    df["first_name_lower"] = df["first_name"].str.strip().str.lower()
    df["last_name_lower"] = df["last_name"].str.strip().str.lower()

    df_names = df_names.copy()
    df_names["first_name_lower"] = df_names["first_name"].str.strip().str.lower()
    df_names["last_name_lower"] = df_names["last_name"].str.strip().str.lower()

    df = df.merge(
        df_names[["first_name_lower", "last_name_lower", "gender", "ethnicity", "name_ID"]],
        on=["first_name_lower", "last_name_lower"],
        how="left",
    )

    rows = []

    for _, row in df.iterrows():
        try:
            j = parse_response_payload(row)
        except Exception:
            continue

        texts = flatten_json(j)
        full_text = " ".join(texts)

        doc = nlp(full_text)
        n_tokens = len(doc) if len(doc) > 0 else 0

        num_nouns = sum(1 for t in doc if t.pos_ == "NOUN")
        num_verbs = sum(1 for t in doc if t.pos_ == "VERB")
        num_adjectives = sum(1 for t in doc if t.pos_ == "ADJ")
        num_adverbs = sum(1 for t in doc if t.pos_ == "ADV")
        num_pronouns = sum(1 for t in doc if t.pos_ == "PRON")
        num_numerals = sum(1 for t in doc if t.pos_ == "NUM")
        num_other = sum(
            1
            for t in doc
            if t.pos_ not in ["NOUN", "VERB", "ADJ", "ADV", "PRON", "NUM"]
        )

        factor = 1000.0 / n_tokens if n_tokens > 0 else 0.0

        rows.append(
            {
                "profile_id": row["profile_id"],
                "first_name": row["first_name"],
                "last_name": row["last_name"],
                "gender": row.get("gender"),
                "ethnicity": row.get("ethnicity"),
                "name_ID": row.get("name_ID"),
                "provider": row.get("provider"),
                "model": row.get("model"),
                "total_tokens": n_tokens,
                "num_nouns": num_nouns,
                "num_verbs": num_verbs,
                "num_adjectives": num_adjectives,
                "num_adverbs": num_adverbs,
                "num_pronouns": num_pronouns,
                "num_numerals": num_numerals,
                "num_other": num_other,
                "nouns_per_1k": num_nouns * factor,
                "verbs_per_1k": num_verbs * factor,
                "adjectives_per_1k": num_adjectives * factor,
                "adverbs_per_1k": num_adverbs * factor,
                "pronouns_per_1k": num_pronouns * factor,
                "numerals_per_1k": num_numerals * factor,
                "other_per_1k": num_other * factor,
            }
        )

    return pd.DataFrame(rows)


In [3]:

# -------------------------------------------------
#                  MAIN
# -------------------------------------------------
def run_pos_counts(model_runs=MODEL_RUNS, out_paths=OUT_PATHS):
    outputs = {}

    for model_name, model_df in model_runs.items():
        df_pos = process_pos(model_df, df_names)
        out_path = out_paths[model_name]
        df_pos.to_csv(out_path, index=False)
        outputs[model_name] = df_pos
        print(f"{model_name}: saved {len(df_pos)} rows to {out_path}")

    return outputs


pos_outputs = run_pos_counts()

# Backwards-compatible variables for later exploratory cells.
df_pos_openai = pos_outputs["openai"]
df_pos_gemini = pos_outputs["gemini"]
df_pos_qwen4B = pos_outputs["qwen4B"]
df_pos_qwen8B = pos_outputs["qwen8B"]


openai: saved 1200 rows to /Users/charlotteleininger/Master/Consulting/CV_analysis/data/pos_counts/openai_cv_pos_counts.csv


<unknown>:76: SyntaxWarning: invalid escape sequence '\*'
<unknown>:110: SyntaxWarning: invalid escape sequence '\*'
<unknown>:56: SyntaxWarning: invalid escape sequence '\*'
<unknown>:7: SyntaxWarning: invalid escape sequence '\*'
<unknown>:97: SyntaxWarning: invalid escape sequence '\*'
<unknown>:131: SyntaxWarning: invalid escape sequence '\*'
<unknown>:147: SyntaxWarning: invalid escape sequence '\*'
<unknown>:145: SyntaxWarning: invalid escape sequence '\*'
<unknown>:72: SyntaxWarning: invalid escape sequence '\*'
<unknown>:86: SyntaxWarning: invalid escape sequence '\*'
<unknown>:57: SyntaxWarning: invalid escape sequence '\*'
<unknown>:105: SyntaxWarning: invalid escape sequence '\*'
<unknown>:73: SyntaxWarning: invalid escape sequence '\*'
<unknown>:140: SyntaxWarning: invalid escape sequence '\*'


gemini: saved 1200 rows to /Users/charlotteleininger/Master/Consulting/CV_analysis/data/pos_counts/gemini_cv_pos_counts.csv
qwen4B: saved 1200 rows to /Users/charlotteleininger/Master/Consulting/CV_analysis/data/pos_counts/qwen4B_cv_pos_counts.csv
qwen8B: saved 1200 rows to /Users/charlotteleininger/Master/Consulting/CV_analysis/data/pos_counts/qwen8B_cv_pos_counts.csv
